### Load csv into pandas dataframe, parse date and time into single timestamp, filter out unimportant columns and low confidence records.

In [ ]:
import pandas as pd
from pathlib import Path

date = "2026-05-02"

file_path = Path.cwd().parent.joinpath(
    "data/raw", f"VIIRS_SNNP_NRT_world_14days_{date}.csv"
)

data = pd.read_csv(file_path)
# TODO remove
# transform time in a string with actual utc mil time format
# data["acq_time"] = data["acq_time"].apply(lambda x: f"{x:04d}")
# data["timestamp"] = pd.to_datetime(
#     data["acq_date"] + data["acq_time"], format="%Y-%m-%d%H%M", utc=True
# )
# data.drop(
#     columns=["acq_date", "satellite", "instrument", "version", "acq_time"], inplace=True
# )

data.drop(columns=["satellite", "instrument", "version", "acq_time"], inplace=True)

data = data[data["confidence"] != "l"]

data.info()
data.head()


### Load other data into geodataframe

In [ ]:
import geopandas as gpd

fire_data = gpd.GeoDataFrame(
    data, geometry=gpd.points_from_xy(data.longitude, data.latitude), crs="EPSG:4326"
)

countries_area = pd.read_csv(Path.cwd().parent.joinpath("data/raw", "surface_area.csv"))

countries_boundaries = gpd.read_file(
    Path.cwd().parent.joinpath("data/raw", "country_polygons.geojson")
).to_crs("EPSG:4326")

countries_boundaries.info()

### Filter out attributes in country area and polygons, cast strings into correct types
Also keep only area data from the last surveyed year (2023).

In [ ]:
countries_area = countries_area[countries_area["TIME_PERIOD"] == 2023]
countries_area = countries_area[["REF_AREA", "OBS_VALUE"]]
countries_area["OBS_VALUE"] = countries_area["OBS_VALUE"].astype(float)
countries_area = countries_area.rename(
    columns={"OBS_VALUE": "country_area", "REF_AREA": "iso_code"}
)
countries_boundaries = countries_boundaries.drop(columns="ISO3166-1-Alpha-2")
countries_area.dropna()
countries_area.info()

### Compute true pixel area for fires pixels

In [ ]:
fire_data["fire_area"] = fire_data["scan"] * fire_data["track"]
fire_data = fire_data[["acq_date", "geometry", "fire_area"]]
fire_data.info()

### Join countries polygons with area table

In [ ]:
countries_boundaries_area = pd.merge(
    countries_boundaries,
    countries_area,
    left_on="ISO3166-1-Alpha-3",
    right_on="iso_code",
    how="left",
)
countries_boundaries_area = countries_boundaries_area.drop(columns="iso_code")
countries_boundaries_area.info()

### Spatially join countries with fires. Merge and group fire data by date and countries.

In [ ]:
fire_data_countries = gpd.sjoin(
    countries_boundaries_area,
    fire_data,
    how="left",
)

# dissolve() times out and crashes the kernel, worked around by using df.groupby() and joining
# back later with the countries geometries on the iso code key
fire_data_by_countries_dates = pd.DataFrame(
    fire_data_countries.groupby(
        by=["ISO3166-1-Alpha-3", "acq_date", "name", "country_area"]
    )["fire_area"]
    .sum()
    .reset_index()
)
fire_data_by_countries = pd.DataFrame(
    fire_data_countries[["ISO3166-1-Alpha-3", "name", "country_area", "fire_area"]]
    .groupby(by=["ISO3166-1-Alpha-3", "name", "country_area"])["fire_area"]
    .sum()
    .reset_index()
)

fire_data_by_countries = gpd.GeoDataFrame(
    fire_data_by_countries.merge(
        countries_boundaries, on=["ISO3166-1-Alpha-3", "name"], how="right"
    )
)

fire_data_by_countries_dates = gpd.GeoDataFrame(
    fire_data_by_countries_dates.merge(
        countries_boundaries, on=["ISO3166-1-Alpha-3", "name"], how="right"
    )
)
fire_data_by_countries.info()

### Compute perentage of wild fires area

In [ ]:
fire_data_by_countries["area_perc"] = (
    fire_data_by_countries["fire_area"] / fire_data_by_countries["country_area"]
) * 100
fire_data_by_countries_dates["area_perc"] = (
    fire_data_by_countries_dates["fire_area"]
    / fire_data_by_countries_dates["country_area"]
) * 100

### Save processed data

In [ ]:
fire_data_by_countries.to_file(
    Path.cwd().parent.joinpath("data/processed", "fire_data_by_countries.gpkg")
)
fire_data_by_countries_dates.to_file(
    Path.cwd().parent.joinpath("data/processed", "fire_data_by_countries_dates.gpkg")
)